# 🤖 Módulo 06 · Introducción a Machine Learning

Una base conceptual sólida: reconocer si un problema es de ML y de qué tipo, preparar features y target, entrenar, evaluar honestamente y detectar overfitting y leakage.

Versión web con conceptos, reconocimiento, razonamiento y proyecto: [https://rosalesluciano.github.io/datalab/](https://rosalesluciano.github.io/datalab/#/m/m06)

## Cómo trabajar este notebook
1. Ejecutá la celda ⚙️ (carga datos y verificador).
2. **Antes de escribir código**, escribí en un comentario cómo lo vas a resolver.
3. Implementá y guardá tu respuesta en `resultado` (o lo que pida el ejercicio).
4. `comprobar()` verifica. Si te trabás: `ver_pista("id", 1)`, después 2 y 3.
5. `ver_solucion("id")` es el último recurso: muestra idea → pseudocódigo → código.

## Conceptos del módulo
- **¿Qué es Machine Learning?**: Construir un modelo que **aprende una relación a partir de datos históricos** para hacer predicciones o encontrar estructura, en lugar de programar las reglas a mano.
- **Tipos de problema**: Reconocer qué tipo de problema tenés, porque define el modelo, la métrica y cómo se evalúa.
- **Dataset, features y target**: Organizar los datos en **X** (variables que el modelo usa para predecir) e **y** (lo que se quiere predecir).
- **Entrenamiento, validación y test**: Evaluar el modelo con datos que **no vio** al entrenar, para estimar cómo funcionará con casos nuevos.
- **Modelo: entrenar y predecir**: Elegir un algoritmo, entrenarlo con ejemplos (fit) y usarlo para predecir casos nuevos (predict).
- **Métricas de regresión**: Medir qué tan lejos están las predicciones numéricas de los valores reales.
- **Métricas de clasificación**: Medir qué tan bien clasifica un modelo y **qué tipo de errores** comete.
- **Overfitting y underfitting**: Detectar si el modelo **memorizó** los datos de entrenamiento (overfitting) o es **demasiado simple** para captar el patrón (underfitting).
- **Variables categóricas**: Convertir categorías (barrio, tipo de contrato) en números que los modelos puedan usar, sin inventar un orden falso.
- **Pipeline conceptual de un proyecto de ML**: Ordenar el trabajo de ML de punta a punta y automatizar los pasos de preprocesamiento + modelo para evitar errores.

In [ ]:
#@title ⚙️ 1) Ejecutá esta celda primero (datos y verificador) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series", list: "una lista", dict: "un diccionario",
               tuple: "una tupla", set: "un set", str: "un texto (str)", int: "un entero (int)",
               float: "un decimal (float)", bool: "un booleano (bool)", type(None): "None (nada)",
               np.ndarray: "un array de NumPy"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    if isinstance(e, dict):
        return (isinstance(u, dict) and set(map(str, u)) == set(map(str, e))
                and all(_scalar_eq(u[k] if k in u else u[str(k)], v) for k, v in e.items()))
    if isinstance(e, (set, frozenset)):
        return isinstance(u, (set, frozenset)) and u == e
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    if u is None:
        return False, ("Tu resultado es None. ¿Te falta un `return` en la función, o guardaste lo que devuelve "
                       "un método que modifica en el lugar (como `.sort()` o `.append()`)?")
    contenedores = (list, tuple, dict, set, str)
    if isinstance(e, contenedores) and not isinstance(u, type(e)) and not (
            isinstance(e, (list, tuple)) and isinstance(u, (list, tuple))):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}."
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple)) and len(u) != len(e):
        return False, f"Tu resultado tiene {len(u)} elementos y se esperaban {len(e)}."
    if isinstance(e, dict) and isinstance(u, dict) and set(map(str, u)) != set(map(str, e)):
        faltan = [k for k in e if str(k) not in set(map(str, u))]
        sobran = [k for k in u if str(k) not in set(map(str, e))]
        msg = "Las claves del diccionario no coinciden."
        if faltan:
            msg += f" Faltan: {faltan}."
        if sobran:
            msg += f" Sobran: {sobran}."
        return False, msg
    txt = repr(u)
    return False, f"Tu resultado es {txt[:300] + ('…' if len(txt) > 300 else '')}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usá & (y), | (o) y poné cada condición entre paréntesis."),
    ("can't multiply sequence", "Estás multiplicando un texto (str). ¿El número llegó como texto? Convertilo con int() o float()."),
    ("can only concatenate str", "Estás sumando texto con otra cosa. Convertí con str() para unir textos, o con float() para hacer cuentas."),
    ("unsupported operand type", "Los dos lados de la operación tienen tipos incompatibles (por ejemplo, int + str o None + float). Revisá type() de cada valor."),
    ("could not convert string to float", "Hay un texto que no representa un número (por ejemplo 'n/d' o ''). Filtralo o capturá el ValueError con try/except."),
    ("invalid literal for int", "Hay un texto que no representa un entero. Revisá el valor o usá try/except ValueError."),
    ("index out of range", "Pediste una posición que no existe. Recordá: las posiciones van de 0 a len()-1, y -1 es el último."),
    ("values to unpack", "La cantidad de variables a la izquierda no coincide con la cantidad de valores que llegan."),
    ("ZeroDivisionError", "Dividiste por cero. ¿La lista o el divisor pueden estar vacíos? Verificalo antes de dividir."),
    ("'NoneType' object", "Un valor es None. Suele pasar con una función sin return o con métodos como .sort() que devuelven None."),
    ("KeyError", "Esa clave o columna no existe. Revisá mayúsculas, tildes y espacios; con dicts podés usar .get()."),
    ("NameError", "Usás una variable que no existe (todavía). ¿Error de tipeo o la definiste más abajo?"),
    ("IndentationError", "Revisá la sangría: el contenido de if, for, def, etc. va con 4 espacios."),
    ("SyntaxError", "Error de sintaxis: revisá paréntesis, comillas, dos puntos (:) al final de if/for/def y que cada línea esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este tipo de objeto. Revisá el nombre y qué tipo tiene la variable."),
    ("MergeError", "Revisá las columnas que usás para unir (on / left_on / right_on)."),
    ("TypeError", "Algún tipo de dato no encaja con la operación. Revisá type() de las variables involucradas."),
    ("ValueError", "El tipo es correcto pero el valor no es válido para esta operación."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\n_rng = np.random.default_rng(42)\n_n = 200\n_m2 = _rng.integers(35, 220, _n)\n_hab = np.clip(_m2 // 35 + _rng.integers(-1, 2, _n), 1, 6)\n_ant = _rng.integers(0, 60, _n)\n_barrio = _rng.choice(["Centro", "Norte", "Sur"], _n)\n_extra = np.select([_barrio == "Centro", _barrio == "Norte"], [40000, 15000], 0)\ncasas = pd.DataFrame({\n    "m2": _m2, "habitaciones": _hab, "antiguedad": _ant, "barrio": _barrio,\n    "precio": np.round(30000 + _m2 * 1800 + _hab * 5000 - _ant * 900 + _extra\n                       + _rng.normal(0, 15000, _n), -2),\n})\n_n = 300\nchurn = pd.DataFrame({\n    "meses_cliente": _rng.integers(1, 72, _n),\n    "cargo_mensual": np.round(_rng.uniform(15, 120, _n), 2),\n    "reclamos": _rng.poisson(1.2, _n),\n    "contrato": _rng.choice(["Mensual", "Anual"], _n, p=[0.6, 0.4]),\n})\n_logit = (-1.0 - 0.05 * churn["meses_cliente"] + 0.03 * churn["cargo_mensual"]\n          + 0.6 * churn["reclamos"] + np.where(churn["contrato"] == "Mensual", 1.2, -0.8))\nchurn["abandona"] = (_rng.uniform(0, 1, _n) < 1 / (1 + np.exp(-_logit))).astype(int)\ndel _rng, _n, _m2, _hab, _ant, _barrio, _extra, _logit\n'
_SOL_B64 = '...'
_TABLAS = ['casas', 'churn']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todos los datos limpios (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    from IPython.display import HTML, display
    import html as _h
    if ok:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">✅ Verificado. Ahora explicá con tus palabras por qué funciona '
                     'y probá una variante.</div>'))
    else:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_pista(ej, nivel=1):
    """Pistas progresivas (1-3). Pedí la siguiente solo si la anterior no alcanzó."""
    pistas = _SOL[ej]["hints"]
    print(f"Pista {nivel}/{len(pistas)}: {pistas[min(nivel, len(pistas)) - 1]}")

def ver_solucion(ej):
    """Último recurso: primero la idea, después el pseudocódigo y al final el código."""
    info = _SOL[ej]
    print("💡 Idea:\n" + info["explain"] + "\n")
    print("🧭 Pseudocódigo:\n" + info["pseudo"] + "\n")
    print("🧑‍💻 Código:\n" + info["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Features, target y división de datos

## 1. ¿Qué es Machine Learning?
`Código` · `Muy fácil`

Preparás un modelo para estimar precios de casas. Guardá en `X` las features `m2`, `habitaciones` y `antiguedad` de `casas`, y en `y` el target `precio`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

X = 
y = 

comprobar("ml01")

## 2. Entrenamiento y prueba
`Código` · `Medio`

Reservá un **20% de los datos para test** (de forma reproducible, `random_state=42`). Guardá `X_test` en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']

# divide aquí

resultado = X_test

comprobar("ml02")

## 3. La feature que sabe demasiado
`Debugging` · `Medio`

Este modelo de churn tiene una accuracy **sospechosamente perfecta**. Encontrá el problema y corregilo para que use solo `meses_cliente`, `cargo_mensual` y `reclamos`.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos', 'abandona']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
arbol = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
resultado = accuracy_score(y_test, arbol.predict(X_test))

comprobar("ml-e01")

## 4. Variables categóricas
`Código` · `Difícil`

El barrio influye en el precio, pero es texto. Convertí la columna `barrio` de `casas` en columnas numéricas 0/1 (one-hot) con `drop_first=True` y `dtype=int`. Guardá el DataFrame completo.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("ml03")

---
# 📌 Regresión: predecir números

## 5. Entrena tu primer modelo
`Código` · `Medio`

Entrená una regresión lineal (`LinearRegression`) con `X_train` e `y_train` y guardá lo que aprendió (sus coeficientes, `modelo.coef_`) en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression

modelo = 
# entrena aquí

resultado = modelo.coef_

comprobar("ml04")

## 6. Hacer predicciones
`Código` · `Medio`

Usá el modelo entrenado para estimar los precios de las casas de `X_test`. Guardá el array de predicciones.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)

resultado = 

comprobar("ml05")

## 7. ¿Qué tan bien predice? MAE
`Código` · `Medio`

¿Cuánto se equivoca el modelo en promedio? Calculá el **MAE** sobre el conjunto de test.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

from sklearn.metrics import mean_absolute_error
resultado = 

comprobar("ml06")

## 8. Evaluar con lo que ya vio
`Debugging` · `Fácil`

Este código calcula el MAE del modelo de precios, pero lo hace sobre los datos equivocados. Corregilo para obtener una evaluación honesta.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
modelo = LinearRegression().fit(X_train, y_train)
resultado = mean_absolute_error(y_train, modelo.predict(X_train))

comprobar("ml-e02")

## 9. Coeficiente R²
`Código` · `Difícil`

Calculá el **R²** del modelo sobre test (`r2_score`) y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

from sklearn.metrics import r2_score
resultado = 

comprobar("ml07")

## 10. Predecir un caso nuevo
`Código` · `Medio`

Un cliente pregunta cuánto valdría una casa de **90 m²**, **3 habitaciones** y **10 años**. Estimá su precio y guardá el **número** (no el array) en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

nueva = pd.DataFrame([{'m2': 90, 'habitaciones': 3, 'antiguedad': 10}])
resultado = 

comprobar("ml08")

## 11. Predecir con una lista plana
`Debugging` · `Fácil`

Este código intenta predecir el precio de una casa nueva, pero falla. Corregilo (devolviendo el número).

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

from sklearn.model_selection import train_test_split

X = casas[['m2', 'habitaciones', 'antiguedad']]
y = casas['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
modelo = LinearRegression().fit(X_train, y_train)
resultado = modelo.predict([90, 3, 10])[0]

comprobar("ml-e03")

## 12. La IA agrega el barrio al modelo
`IA copiloto` · `Difícil`

Querías mejorar el modelo de precios agregando el barrio. La IA te dio este código. Analizalo antes de confiar en el resultado.

> 🤖 Prompt usado:
> Tengo el DataFrame casas (m2, habitaciones, antiguedad, barrio, precio). Quiero una regresión lineal que use también el barrio. Explicá cómo tratás la variable categórica y cómo evaluás.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

datos = pd.get_dummies(casas, columns=['barrio'], drop_first=True, dtype=int)
X = datos.drop(columns=['precio'])
y = datos['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modelo = LinearRegression().fit(X_train, y_train)
resultado = mean_absolute_error(y_test, modelo.predict(X_test))

comprobar("ml-f01")

---
# 📌 Clasificación: predecir categorías

## 13. El baseline: ¿cuánto acierta no hacer nada?
`Código` · `Fácil`

Antes de entrenar un clasificador de churn, calculá la **accuracy del baseline**: predecir siempre la clase más frecuente en `y_test`. Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

resultado = 

comprobar("ml-d01")

## 14. Regresión logística
`Código` · `Difícil`

Entrená un clasificador `LogisticRegression(max_iter=1000)`, predecí sobre `X_test` y guardá la **accuracy**. Después compará mentalmente con el baseline.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

resultado = 

comprobar("ml09")

## 15. Matriz de confusión
`Código` · `Difícil`

¿Qué tipo de errores comete el modelo logístico? Calculá su **matriz de confusión** y guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)

from sklearn.metrics import confusion_matrix
resultado = 

comprobar("ml10")

## 16. Árbol de decisión
`Código` · `Difícil`

Probá otro algoritmo: entrená un `DecisionTreeClassifier(max_depth=3, random_state=0)` y guardá su **accuracy** en test.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

resultado = 

comprobar("ml11")

---
# 📌 Overfitting, pipelines y validación

## 17. Transferencia: el árbol sin límites
`Transferencia` · `Difícil`

Entrená un `DecisionTreeClassifier(random_state=0)` **sin max_depth** y guardá en `resultado` una lista `[accuracy_train, accuracy_test]`.

> 🔁 **Qué cambia:** Antes entrenaste un árbol limitado a profundidad 3. Ahora vas a sacar el límite y comparar **train vs test** para ver qué pasa.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

resultado = 

comprobar("ml-g01")

## 18. Escalado y Pipeline
`Código` · `Desafío`

Armá un pipeline con `StandardScaler()` y `KNeighborsClassifier(n_neighbors=5)`, entrenalo y guardá su **accuracy** en test.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

pipe = 
resultado = 

comprobar("ml12")

## 19. Validación cruzada
`Código` · `Desafío`

Evaluá un `DecisionTreeClassifier(max_depth=3, random_state=0)` con **validación cruzada de 5 folds** sobre `X` e `y` completos y guardá la **media** de los scores.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']

resultado = 

comprobar("ml13")

---
# 📌 Proyecto: predecir el abandono de clientes

## 20. Fase 2 · El baseline del proyecto
`Código` · `Medio`

Prepará los datos según el plan (one-hot de `contrato` con `drop_first=True` y `dtype=int`; X sin `abandona`), dividí 75/25 estratificado con `random_state=42` y guardá en `resultado` la **accuracy del baseline** (clase mayoritaria de `y_test`).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split

resultado = 

comprobar("ml-p2")

## 21. Fase 3 · Entrenar el modelo
`Código` · `Desafío`

Con los datos preparados como en la fase 2, entrená un `RandomForestClassifier(n_estimators=100, random_state=42)` y guardá su **accuracy** en test. (El código de la solución también imprime la importancia de cada variable.)

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Preparar

# 2. Dividir

# 3. Entrenar

# 4. Evaluar
resultado = 

comprobar("ml14")

## 22. Fase 4 · El escalado que espía
`Debugging` · `Difícil`

Un compañero escaló los datos **antes** de dividirlos para entrenar un KNN. Funciona, pero es una mala práctica que filtra información del test. Corregilo usando un pipeline que escale **solo con train**.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X = churn[['meses_cliente', 'cargo_mensual', 'reclamos']]
y = churn['abandona']
X_esc = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_esc, y, test_size=0.25, random_state=42, stratify=y)
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
resultado = accuracy_score(y_test, knn.predict(X_test))

comprobar("ml-p4")

---
# 🏁 Fin del notebook
Volvé a la [plataforma](https://rosalesluciano.github.io/datalab/) para los ejercicios de reconocimiento, razonamiento y el modo **¿Sé esto?**.